# Drift Control — end-to-end (new architecture)

A single pass through the staged API: **validate → detect → report → adapt**.
Everything here uses stable public names from `drift_control` (see `ROADMAP.md`).

In [ ]:
import numpy as np

from drift_control import (
    DDM,
    PerformanceDriftMonitor,
    TriggerRetrainingPolicy,
    UnivariateDriftDetector,
    binary_segmentation,
    validate_reference_current,
)
from drift_control.monitoring import DriftReport

rng = np.random.default_rng(0)

## 1. Validate, then batch feature-wise data drift

`validate_reference_current` gives a clean common representation; `UnivariateDriftDetector` scores each feature with multiple-testing correction.

In [ ]:
ref = rng.normal(0, 1, size=(500, 3))
cur = rng.normal(0, 1, size=(500, 3))
cur[:, 1] = rng.normal(1.5, 1, size=500)  # only feature 'b' drifts

pair = validate_reference_current(ref, cur, feature_names=['a', 'b', 'c'])
detector = UnivariateDriftDetector(method='ks', feature_names=['a', 'b', 'c']).fit(pair.reference)
result = detector.detect(pair.current)
print('aggregate drift:', result.drift)
print('drifting features:', result.metadata['drifting_features'])

## 2. Aggregate the per-feature results into a report

In [ ]:
report = DriftReport.from_results(detector.detect_features(pair.current))
print(report.to_markdown())

## 3. Online concept drift over an error stream

`DDM` consumes a per-item error indicator and flags when the error rate jumps.

In [ ]:
ddm = DDM(min_samples=30)
errors = np.concatenate([rng.random(300) < 0.1, rng.random(300) < 0.6]).astype(float)
flagged_at = next((i for i, e in enumerate(errors) if ddm.update(float(e)).drift_detected), None)
print('concept drift flagged at step:', flagged_at)

## 4. Change point in a time-indexed signal

In [ ]:
series = np.concatenate([rng.normal(0, 0.3, 150), rng.normal(3, 0.3, 150)])
segments = binary_segmentation(series, penalty=50.0)
print('change points:', segments.change_points)

## 5. Performance drift drives a retraining decision

A `PerformanceDriftMonitor` measures the degradation; a `RetrainingPolicy` decides what to do.

In [ ]:
monitor = PerformanceDriftMonitor(metrics=['accuracy'], window=100, reference={'accuracy': 0.95})
y_true = (rng.random(100) < 0.5).astype(int)
y_pred = y_true.copy()
y_pred[:40] = 1 - y_pred[:40]  # 60% accuracy now
monitor.update(y_true, y_pred)
perf = monitor.detect()

policy = TriggerRetrainingPolicy(
    required_drift_events=1, metric='accuracy', min_metric_drop=0.1, baseline_metric=0.95,
)
print('performance drift:', perf.drift)
print('should retrain? ', policy.should_retrain(perf, perf.metadata['current']))